In [1]:
# last updated 2025-05-13 by mza
# from https://keras.io/keras_tuner/ and https://keras.io/keras_tuner/getting_started/
name = "mza_try11"
truths_to_use = [ 0, 1, 2, 3 ] # t_peak, t_sigma, height, pedestal
hidden_node_activation_options = [ "relu", "tanh", "sigmoid" ]
#hidden_node_activation_options = [ "sigmoid" ]
seed = 7
max_trials = 30
executions_per_trial = 2
num_epochs = 25
#objective = "val_accuracy"
objective = "val_loss"
#loss = "categorical_crossentropy"
loss = "mse"
#optimizer = "adam"
optimizer = "sgd"
min_layers = 4
max_layers = 5
min_nodes = 32; max_nodes = 512; step_nodes = 32
learning_rate_min = 0.001
learning_rate_max = 0.1

In [2]:
import numpy as np
dataset1 = np.load('waveform_data_0.npy') # (10000, 104)
dataset2 = np.load('waveform_data_1.npy') # (10000, 104)
dataset = np.concatenate((dataset1, dataset2))
num_waveforms = len(dataset)
training_quantity = int(0.8 * num_waveforms)
num_truths = 4
time_samples = len(dataset[0]) - num_truths
#print("num_waveforms: " + str(num_waveforms))
num_truths_to_use = len(truths_to_use)
#print("num_truths_to_use: " + str(num_truths_to_use))
waveforms = dataset[:,num_truths:]
truths = dataset[:,:num_truths]
waveform_min = min([ min(waveforms[i]) for i in range(len(waveforms)) ])
waveform_max = max([ max(waveforms[i]) for i in range(len(waveforms)) ])
offset = waveform_min
gain = 1.0 / (waveform_max - waveform_min)
scaled_waveforms = np.array([ [ gain * (waveforms[j,i] - offset) for i in range(time_samples) ] for j in range(num_waveforms) ])
scaled_truths = np.array([ [ truths[j,0]/time_samples, truths[j,1]/time_samples, gain * (truths[j,2] - offset), gain * (truths[j,3] - offset) ] for j in range(num_waveforms) ])
train_data = scaled_waveforms[:training_quantity,:]
#print("train_data.shape: " + str(train_data.shape))
train_truth = scaled_truths[:training_quantity,truths_to_use]
#print("train_truth.shape: " + str(train_truth.shape))
test_data = scaled_waveforms[training_quantity:,:]
#print("test_data.shape: " + str(test_data.shape))
test_truth = scaled_truths[training_quantity:,truths_to_use]
#print("test_truth.shape: " + str(test_truth.shape))

In [3]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import keras, keras_tuner, keras.layers
def myoptimizer(learning_rate):
    if optimizer == "adam":
        return keras.optimizers.Adam(learning_rate=learning_rate)
    if optimizer == "sgd":
        return keras.optimizers.SGD(learning_rate=learning_rate)
def build_model(hp):
    model = keras.Sequential()
    for i in range(hp.Int("num_layers", min_layers, max_layers)):
        model.add(
            keras.layers.Dense(
                units=hp.Int(f"units_{i}", min_value=min_nodes, max_value=max_nodes, step=step_nodes),
                activation=hp.Choice("activation", hidden_node_activation_options),
            )
        )
    model.add(keras.layers.Dense(num_truths_to_use, activation="sigmoid"))
    learning_rate = hp.Float("lr", min_value=learning_rate_min, max_value=learning_rate_max, sampling="log")
    model.compile(optimizer=myoptimizer(learning_rate), loss=loss, metrics=["accuracy"])
    return model

In [4]:
dirname = name + "." + str(max_trials) + "-" + str(executions_per_trial) + "-" + str(seed)
tuner = keras_tuner.RandomSearch(hypermodel=build_model, objective=objective,
    seed=seed, max_trials=max_trials, executions_per_trial=executions_per_trial,
    overwrite=True, project_name=name, directory=dirname)

In [5]:
tuner.search(train_data, train_truth, epochs=num_epochs, validation_data=(test_data, test_truth))

Trial 30 Complete [00h 00m 39s]
val_loss: 0.010695422068238258

Best val_loss So Far: 0.0012822630815207958
Total elapsed time: 00h 34m 30s
INFO:tensorflow:Oracle triggered exit


In [6]:
#models = tuner.get_best_models(num_models=2)
#best_model = models[0]
#best_model.summary()
best_hp = tuner.get_best_hyperparameters()[0]
print(str(best_hp.values))

{'num_layers': 4, 'units_0': 224, 'activation': 'relu', 'units_1': 320, 'units_2': 96, 'units_3': 512, 'lr': 0.08316141310476274, 'units_4': 448}


In [7]:
model = build_model(best_hp)
model(test_data)
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_5 (Dense)             (4000, 224)               22624     
                                                                 
 dense_6 (Dense)             (4000, 320)               72000     
                                                                 
 dense_7 (Dense)             (4000, 96)                30816     
                                                                 
 dense_8 (Dense)             (4000, 512)               49664     
                                                                 
 dense_9 (Dense)             (4000, 4)                 2052      
                                                                 
Total params: 177156 (692.02 KB)
Trainable params: 177156 (692.02 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [8]:
#import hypermodel # pip install hypermodel

In [9]:
#from keras_tuner import HyperModel